In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

file_path = '/content/drive/MyDrive/Week4/MSCI_world_CIK.xlsx'
df = pd.read_excel(file_path)

In [3]:
df

,Security Name,Closing Weight,Ticker,CIK,Bloomberg_Ticker,Country,SEC_name,Similarity_Check
0,NVIDIA,0.053269,NVDA,1045810.0,NVDA US Equity,United States,NVIDIA CORP,Similar
1,APPLE,0.050672,AAPL,320193.0,AAPL US Equity,United States,Apple Inc.,Similar
2,MICROSOFT CORP,0.041523,MSFT,789019.0,MSFT US Equity,United States,MICROSOFT CORP,Similar
3,AMAZON,0.027176,AMZN,1018724.0,AMZN US Equity,United States,AMAZON COM INC,Similar
4,ALPHABET A,0.023194,GOOGL,1652044.0,GOOGL US Equity,United States,Alphabet Inc.,Similar
...,...,...,...,...,...,...,...,...
1316,HOLMEN B,0.000032,JBHT,728535.0,JBHT US Equity,NaN,HUNT J B TRANSPORT SERVICES INC,Not Similar (Score: 26)
1317,INFRASTRUTTURE WIRELESS,0.000031,INW.MI,NaN,INW IM Equity,Italy,NaN,No SEC Name to Compare
1318,NEXI,0.000030,SCYX,1178253.0,SCYX US Equity,NaN,SCYNEXIS INC,Not Similar (Score: 50)
1319,JD SPORTS FASHION,0.000030,JD,1549802.0,JD US Equity,NaN,"JD.com, Inc.",Not Similar (Score: 37)


In [4]:
import requests
import time
import pandas as pd
from tqdm.notebook import tqdm

headers = {'User-Agent': 'ResearchProject user@example.com'}

def process_filings_dict(filings_dict):
    if not filings_dict:
        return 0
    df_f = pd.DataFrame(filings_dict)
    if df_f.empty or 'form' not in df_f.columns or 'filingDate' not in df_f.columns:
        return 0

    valid_forms = ['10-K', '20-F']
    mask_form = df_f['form'].isin(valid_forms)
    df_f['filingDate'] = pd.to_datetime(df_f['filingDate'])
    mask_year = (df_f['filingDate'].dt.year >= 2014) & (df_f['filingDate'].dt.year <= 2025)

    return len(df_f[mask_form & mask_year])

def count_filings_comprehensive(cik):
    if pd.isna(cik):
        return 0

    cik_str = str(int(cik)).zfill(10)
    base_url = "https://data.sec.gov/submissions/"
    file_name = f"CIK{cik_str}.json"
    url = f"{base_url}{file_name}"

    count = 0
    try:
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            print(f"{file_name} -> count: 0")
            return 0
        data = response.json()

        # 1. Process recent filings (up to 1000 most recent)
        recent_filings = data.get('filings', {}).get('recent', {})
        count += process_filings_dict(recent_filings)

        # 2. Process older archived filings if they exist
        older_files = data.get('filings', {}).get('files', [])
        for file_info in older_files:
            # Only fetch the older file if it potentially contains data from 2014 or later
            to_year = int(str(file_info.get('filingTo', '0'))[:4])
            if to_year >= 2014:
                time.sleep(0.11)  # Respect rate limit for extra calls
                older_url = f"{base_url}{file_info['name']}"
                older_resp = requests.get(older_url, headers=headers)
                if older_resp.status_code == 200:
                    count += process_filings_dict(older_resp.json())

        print(f"{file_name} -> count: {count}")
        return count
    except Exception as e:
        print(f"{file_name} -> count: 0 (Error: {e})")
        return 0

def process_cik(cik):
    time.sleep(0.11)  # Sleep to respect the limit
    return count_filings_comprehensive(cik)

tqdm.pandas(desc="Fetching SEC Filings (Comprehensive)")
df['10K_20F_Count'] = df['CIK'].progress_apply(process_cik)

display(df[['Security Name', 'Ticker', 'CIK', '10K_20F_Count']])

Fetching SEC Filings (Comprehensive):   0%|          | 0/1321 [00:00<?, ?it/s]

CIK0001045810.json -> count: 12
CIK0000320193.json -> count: 12
CIK0000789019.json -> count: 12
CIK0001018724.json -> count: 12
CIK0001652044.json -> count: 10
CIK0001730168.json -> count: 8
CIK0001652044.json -> count: 10
CIK0001326801.json -> count: 12
CIK0001318605.json -> count: 12
CIK0000059478.json -> count: 12
CIK0000019617.json -> count: 12
CIK0001067983.json -> count: 12
CIK0001403161.json -> count: 12
CIK0000200406.json -> count: 12
CIK0000034088.json -> count: 12
CIK0000104169.json -> count: 12
CIK0001141391.json -> count: 12
CIK0001065280.json -> count: 12
CIK0001551152.json -> count: 12
CIK0000909832.json -> count: 12
CIK0000937966.json -> count: 12
CIK0001321655.json -> count: 5
CIK0000070858.json -> count: 12
CIK0000354950.json -> count: 12
CIK0000080424.json -> count: 12
CIK0001341439.json -> count: 12
CIK0000002488.json -> count: 12
CIK0000040545.json -> count: 12
CIK0000858877.json -> count: 12
CIK0000021344.json -> count: 12
CIK0000731766.json -> count: 12
CIK0000901

,Security Name,Ticker,CIK,10K_20F_Count
0,NVIDIA,NVDA,1045810.0,12
1,APPLE,AAPL,320193.0,12
2,MICROSOFT CORP,MSFT,789019.0,12
3,AMAZON,AMZN,1018724.0,12
4,ALPHABET A,GOOGL,1652044.0,10
...,...,...,...,...
1316,HOLMEN B,JBHT,728535.0,12
1317,INFRASTRUTTURE WIRELESS,INW.MI,NaN,0
1318,NEXI,SCYX,1178253.0,11
1319,JD SPORTS FASHION,JD,1549802.0,11


In [6]:
# 827 names that have 8 or more regulatory filings (10-K/20-F) between 2014 and 2025
df_filtered = df[df['10K_20F_Count'] >= 8]
len(df_filtered)

827

In [7]:
# drop 262 duplicate names - could be companies under same holding or just incorrect mapping
# 565 names narrowed
df_sorted = df_filtered.reset_index().sort_values(by=['CIK', 'index'])
df_unique = df_sorted.drop("index", axis=1).drop_duplicates(subset=['CIK'], keep='first')
len(df_unique)

565

In [8]:
import requests
import time
import pandas as pd
from tqdm.notebook import tqdm

headers = {'User-Agent': 'ResearchProject user@example.com'}

def extract_files_from_dict(filings_dict):
    if not filings_dict:
        return []
    df_f = pd.DataFrame(filings_dict)
    if df_f.empty or 'form' not in df_f.columns or 'filingDate' not in df_f.columns:
        return []

    valid_forms = ['10-K', '20-F']
    df_f['filingDate'] = pd.to_datetime(df_f['filingDate'])
    mask_form = df_f['form'].isin(valid_forms)
    mask_year = (df_f['filingDate'].dt.year >= 2014) & (df_f['filingDate'].dt.year <= 2025)

    filtered = df_f[mask_form & mask_year]

    files = []
    for _, row in filtered.iterrows():
        acc_num = str(row['accessionNumber']).replace('-', '')
        doc = str(row['primaryDocument'])
        files.append(f"{acc_num}/{doc}")

    return files

def get_filing_files(cik):
    if pd.isna(cik):
        return []

    cik_str = str(int(cik)).zfill(10)
    base_url = "https://data.sec.gov/submissions/"
    url = f"{base_url}CIK{cik_str}.json"

    files_list = []
    try:
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            return []
        data = response.json()

        recent = data.get('filings', {}).get('recent', {})
        files_list.extend(extract_files_from_dict(recent))

        older_files = data.get('filings', {}).get('files', [])
        for file_info in older_files:
            to_year = int(str(file_info.get('filingTo', '0'))[:4])
            if to_year >= 2014:
                time.sleep(0.11)
                older_url = f"{base_url}{file_info['name']}"
                older_resp = requests.get(older_url, headers=headers)
                if older_resp.status_code == 200:
                    files_list.extend(extract_files_from_dict(older_resp.json()))

        return files_list
    except Exception as e:
        return []

tqdm.pandas(desc="Fetching Filing Files")

def process_cik_files(cik):
    time.sleep(0.11)  # SEC rate limit (10 requests/second)
    return get_filing_files(cik)

df_unique = df_unique.copy()
df_unique['files'] = df_unique['CIK'].progress_apply(process_cik_files)

display(df_unique[['Security Name', 'CIK', '10K_20F_Count', 'files']].head(10))


Fetching Filing Files:   0%|          | 0/565 [00:00<?, ?it/s]

,Security Name,CIK,10K_20F_Count,files
45,ABBOTT LABORATORIES,1800.0,12,"[000162828025007110/abt-20241231.htm, 00016282..."
25,ADVANCED MICRO DEVICES,2488.0,12,"[000000248825000012/amd-20241228.htm, 00000024..."
268,CHENIERE ENERGY,3570.0,12,"[000000357025000033/lng-20241231.htm, 00000035..."
153,HOWMET AEROSPACE,4281.0,12,"[000000428125000011/hwm-20241231.htm, 00000042..."
49,AMERICAN EXPRESS,4962.0,12,"[000000496225000016/axp-20241231.htm, 00000049..."
217,AFLAC,4977.0,12,"[000000497725000047/afl-20241231.htm, 00000049..."
99,ANALOG DEVICES,6281.0,12,"[000000628125000153/adi-20251101.htm, 00000062..."
52,APPLIED MATERIALS,6951.0,12,"[000162828025056742/amat-20251026.htm, 0000006..."
329,ARCH CAPITAL GROUP,7084.0,12,"[000000708425000011/adm-20241231.htm, 00000070..."
704,ASSOCIATED BRITISH FOODS,7789.0,12,"[000000778925000013/asb-20241231.htm, 00000077..."


In [9]:
df_unique = df_unique[df_unique['files'].apply(len) >= 8]
len(df_unique)

565

In [ ]:
import os
import re
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Define a regex pattern to identify geographic dimensions flexibly.
GEO_DIMENSION_PATTERN = re.compile(r'(Geograph|Country|Region).*Axis', re.IGNORECASE)

# Globals for SEC API
SEC_HEADERS = {
    'User-Agent': 'ColabResearchData research_xyz@example.edu',
    'Accept-Encoding': 'gzip, deflate'
}
TICKER_TO_CIK = {}

# Fallback mapping for custom folder names or foreign tickers not in the standard SEC JSON
CUSTOM_TICKER_MAP = {
    'STMMI': '0000466259',
    'ERICB': '0000730466',
    'NOVN': '0001370368',
    'RACE': '0001648416',
    'STLAM': '0001605484',
    'LOGN': '0001032975',
    'NOKIA': '0000924613',
    'ASML': '0000937966',
    'SAP': '0001104659'
}

def sec_get_with_retry(url, max_retries=8):
    """Helper to fetch from SEC with exponential backoff for 429/503 limits."""
    for i in range(max_retries):
        time.sleep(0.15)  # Base limit
        resp = requests.get(url, headers=SEC_HEADERS)
        if resp.status_code == 200:
            return resp
        elif resp.status_code in [429, 503]:
            sleep_time = 2 ** i  # 1s, 2s, 4s, 8s, 16s
            print(f"      -> Rate limit/Block ({resp.status_code}). Retrying in {sleep_time}s...")
            time.sleep(sleep_time)
        else:
            return resp
    return resp

def get_cik_from_ticker(ticker):
    """Fetches and caches the SEC company tickers JSON to map a Ticker to a CIK."""
    ticker_upper = ticker.upper()

    if ticker_upper in CUSTOM_TICKER_MAP:
        return CUSTOM_TICKER_MAP[ticker_upper]

    global TICKER_TO_CIK
    if not TICKER_TO_CIK:
        try:
            resp = sec_get_with_retry("https://www.sec.gov/files/company_tickers.json")
            if resp.status_code == 200:
                for item in resp.json().values():
                    TICKER_TO_CIK[item['ticker'].upper()] = str(item['cik_str'])
        except Exception as e:
            print(f"Error fetching tickers: {e}")
            return None

    return TICKER_TO_CIK.get(ticker_upper)

def fetch_missing_xbrl(file_path):
    """Attempts to download the missing XBRL .xml file from SEC EDGAR based on the filename."""
    filename = os.path.basename(file_path)

    accession_match = re.search(r'(\d{10}-\d{2}-\d{6})', filename)
    if not accession_match:
        print(f"    -> Could not find accession number in filename: {filename}")
        return None, None

    accession_number = accession_match.group(1)
    accession_clean = accession_number.replace('-', '')

    cik = None

    try:
        with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
            header_text = f.read(10000)
            cik_match = re.search(r'CENTRAL INDEX KEY:\s*0*(\d+)', header_text)
            if cik_match:
                cik = cik_match.group(1)
    except Exception as e:
        pass

    if not cik:
        ticker_candidate = filename.split('_')[0]
        cik = get_cik_from_ticker(ticker_candidate)

    if not cik:
        parent_dir = os.path.basename(os.path.dirname(file_path))
        cik = get_cik_from_ticker(parent_dir)

    if not cik:
        print(f"    -> Could not resolve CIK for file: {file_path}. Consider adding to CUSTOM_TICKER_MAP.")
        return None, None

    cik_clean = str(int(cik))
    index_url = f"https://www.sec.gov/Archives/edgar/data/{cik_clean}/{accession_clean}/index.json"

    try:
        response = sec_get_with_retry(index_url)
        if response.status_code != 200:
            print(f"    -> Failed to load index.json: {index_url} (Status: {response.status_code})")
            return None, None

        index_data = response.json()
        xbrl_file_name = None
        for item in index_data['directory']['item']:
            name = item['name']
            if name.endswith('.xml') and not any(name.endswith(suffix) for suffix in ['_def.xml', '_cal.xml', '_lab.xml', '_pre.xml']):
                xbrl_file_name = name
                break

        if xbrl_file_name:
            xbrl_url = f"https://www.sec.gov/Archives/edgar/data/{cik_clean}/{accession_clean}/{xbrl_file_name}"
            print(f"    -> Downloading missing XBRL from SEC: {xbrl_file_name}")
            xbrl_response = sec_get_with_retry(xbrl_url)

            if xbrl_response.status_code == 200:
                base, _ = os.path.splitext(file_path)
                new_file_path = base + '.xml'
                with open(new_file_path, 'wb') as f:
                    f.write(xbrl_response.content)
                return new_file_path, xbrl_response.content.decode('utf-8', errors='replace')
            else:
                print(f"    -> Failed to download instance: {xbrl_url}")
        else:
             print(f"    -> Could not identify an instance .xml file in the directory.")
    except Exception as e:
        print(f"    -> Error fetching XBRL: {e}")

    return None, None

def parse_single_filing(file_path):
    """
    Parses a single SEC 10-K iXBRL.txt file or traditional XBRL .xml file.
    Returns a list of extracted facts and a list of matched dimensions metadata.
    """
    filename = os.path.basename(file_path)
    ticker = os.path.basename(os.path.dirname(file_path))
    if '_' in filename:
        ticker = filename.split('_')[0]

    file_facts_list = []
    matched_dimensions_list = []

    try:
        with open(file_path, 'r', encoding='utf-8', errors='replace') as file:
            content = file.read()
    except Exception as e:
        print(f"Error reading file {filename}: {e}")
        return file_facts_list, matched_dimensions_list

    print(f"\n====================================================================")
    print(f"Parsing iXBRL/XBRL document: {filename}...")

    if not re.search(r'<(xbrli:)?context', content, re.IGNORECASE):
        print("  -> No XBRL contexts found in this file. Attempting to download missing XML instance...")
        new_path, new_content = fetch_missing_xbrl(file_path)
        if new_content:
            file_path = new_path
            filename = os.path.basename(file_path)
            content = new_content
            print(f"  -> Successfully downloaded and loaded {filename}!")
        else:
            print("  -> Could not fetch XML instance. Skipping.")
            return file_facts_list, matched_dimensions_list

    soup = BeautifulSoup(content, 'lxml-xml')
    geo_contexts_map = {}

    contexts = soup.find_all(re.compile(r'(xbrli:)?context', re.IGNORECASE))

    for context in contexts:
        context_id = context.get('id')
        if not context_id:
            continue

        explicit_members = context.find_all(re.compile(r'(xbrldi:)?explicitMember', re.IGNORECASE))

        is_geo_context = False
        geo_region = None
        matched_dimension = None

        for member in explicit_members:
            dimension = member.get('dimension', '')
            if GEO_DIMENSION_PATTERN.search(dimension):
                is_geo_context = True
                geo_region = member.text.strip()
                matched_dimension = dimension
                matched_dimensions_list.append({
                    'Source File': filename,
                    'Context ID': context_id,
                    'Dimension': dimension,
                    'Region': geo_region
                })
                break

        if is_geo_context:
            period_tag = context.find(re.compile(r'(xbrli:)?period', re.IGNORECASE))
            period_str = "Unknown Period"

            if period_tag:
                instant = period_tag.find(re.compile(r'(xbrli:)?instant', re.IGNORECASE))
                if instant:
                    period_str = f"As of {instant.text.strip()}"
                else:
                    start = period_tag.find(re.compile(r'(xbrli:)?startDate', re.IGNORECASE))
                    end = period_tag.find(re.compile(r'(xbrli:)?endDate', re.IGNORECASE))
                    if start and end:
                        period_str = f"{start.text.strip()} to {end.text.strip()}"

            geo_contexts_map[context_id] = {
                'Region': geo_region,
                'Period': period_str,
                'Dimension': matched_dimension
            }

    facts = soup.find_all(lambda tag: tag.has_attr('contextRef') or tag.has_attr('contextref'))

    for fact in facts:
        context_ref = fact.get('contextRef') or fact.get('contextref')

        if context_ref in geo_contexts_map:
            prefix = fact.prefix + ':' if fact.prefix else ''
            metric_name = fact.get('name') or (prefix + fact.name)
            raw_value = fact.text.strip().replace(',', '')

            try:
                numeric_value = float(raw_value)
            except ValueError:
                continue

            sign = fact.get('sign', '')
            if sign == '-':
                numeric_value = -numeric_value

            scale = fact.get('scale', '0')
            try:
                scale_int = int(scale)
                actual_value = numeric_value * (10 ** scale_int)
            except ValueError:
                actual_value = numeric_value

            unit_ref = fact.get('unitRef') or fact.get('unitref') or 'Unknown Unit'

            file_facts_list.append({
                'Ticker': ticker,
                'Source File': filename,
                'Matched Dimension': geo_contexts_map[context_ref]['Dimension'],
                'Metric': metric_name,
                'Geographic Region (Raw XBRL)': geo_contexts_map[context_ref]['Region'],
                'Reporting Period': geo_contexts_map[context_ref]['Period'],
                'Reported Value': raw_value,
                'Calculated Absolute Value': actual_value,
                'Unit': unit_ref
            })

    return file_facts_list, matched_dimensions_list

def extract_geographic_segment_data(file_paths_list):
    """
    Iterates through a list of SEC 10-K files (.txt or .xml), parses them via a helper function,
    displays data iteratively, and returns unified DataFrames.
    """
    master_data_list = []
    master_matched_dims_list = []

    for file_path in file_paths_list:
        if not os.path.isfile(file_path) or not (file_path.endswith('.txt') or file_path.endswith('.xml')):
            continue

        file_facts, file_dims = parse_single_filing(file_path)

        master_data_list.extend(file_facts)
        master_matched_dims_list.extend(file_dims)

        if file_facts:
            file_df = pd.DataFrame(file_facts)
            for dim, dim_group in file_df.groupby('Matched Dimension'):
                print(f"\n  ---> Numerical Contents for Matched Dimension: '{dim}'")
                display(dim_group.reset_index(drop=True))
        else:
            print("  ---> No numerical geographic facts found for any matched dimension in this file.")

    df = pd.DataFrame(master_data_list) if master_data_list else pd.DataFrame()
    matched_dims_df = pd.DataFrame(master_matched_dims_list) if master_matched_dims_list else pd.DataFrame()

    if df.empty:
        print("\nWarning: No geographic segment data was extracted across all files.")

    return df, matched_dims_df


In [ ]:
import os
import re
import time
import requests
import pandas as pd
from tqdm.notebook import tqdm

os.makedirs('sec_filings_all', exist_ok=True)
headers = {'User-Agent': 'AcademicResearch (research@example.edu)'}

# intermittent saves and checkpoints in case something breaks
facts_csv_path = '/content/drive/MyDrive/sec_extracted_facts.csv'
dims_csv_path = '/content/drive/MyDrive/sec_extracted_dims.csv'
checkpoint_file = '/content/drive/MyDrive/sec_extraction_checkpoint.txt'

processed_ciks = set()
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, 'r') as f:
        processed_ciks = set(line.strip() for line in f)

all_facts_batch = []
all_dims_batch = []
companies_processed_in_batch = 0

def save_batch(facts_batch, dims_batch):
    """Appends the current batch of extracted data to the CSV files."""
    if facts_batch:
        df_f = pd.DataFrame(facts_batch)
        if 'Reporting Period' in df_f.columns:
            df_f['Year'] = df_f['Reporting Period'].apply(
                lambda x: re.findall(r'\d{4}', str(x))[-1] if pd.notnull(x) and re.findall(r'\d{4}', str(x)) else None
            )
        df_f.to_csv(facts_csv_path, mode='a', header=not os.path.exists(facts_csv_path), index=False)

    if dims_batch:
        df_d = pd.DataFrame(dims_batch)
        df_d.to_csv(dims_csv_path, mode='a', header=not os.path.exists(dims_csv_path), index=False)

for _, row in tqdm(df_unique.iterrows(), total=len(df_unique), desc="Processing Filings (2014-2025)"):
    cik = str(int(row['CIK'])).zfill(10)

    if cik in processed_ciks:
        continue

    files_to_process = row['files']

    for file_path in files_to_process:
        url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{file_path}"

        acc_num_raw = file_path.split('/')[0]
        if len(acc_num_raw) == 18:
            acc_num_dashed = f"{acc_num_raw[:10]}-{acc_num_raw[10:12]}-{acc_num_raw[12:]}"
        else:
            acc_num_dashed = acc_num_raw

        filename = file_path.split('/')[-1]
        local_path = os.path.join('sec_filings_all', f"{row['Ticker']}_{acc_num_dashed}_{filename}")

        if not os.path.exists(local_path):
            time.sleep(0.15)
            resp = requests.get(url, headers=headers)
            if resp.status_code == 200:
                with open(local_path, 'wb') as f:
                    f.write(resp.content)
            else:
                print(f"Failed to download {url}")
                continue

        facts, dims = parse_single_filing(local_path)
        all_facts_batch.extend(facts)
        all_dims_batch.extend(dims)

    with open(checkpoint_file, 'a') as f:
        f.write(f"{cik}\n")
    processed_ciks.add(cik)
    companies_processed_in_batch += 1

    if companies_processed_in_batch >= 10:
        print(f"\n--- Saving checkpoint after 10 companies ---")
        save_batch(all_facts_batch, all_dims_batch)
        all_facts_batch.clear()
        all_dims_batch.clear()
        companies_processed_in_batch = 0

if all_facts_batch or all_dims_batch:
    print(f"\n--- Saving final batch ---")
    save_batch(all_facts_batch, all_dims_batch)
    all_facts_batch.clear()
    all_dims_batch.clear()

if os.path.exists(facts_csv_path):
    df_facts = pd.read_csv(facts_csv_path)
    df_dims = pd.read_csv(dims_csv_path) if os.path.exists(dims_csv_path) else pd.DataFrame()
    print("\n--- Extracted Facts (First 10) ---")
    display(df_facts.head(10))
else:
    df_facts = pd.DataFrame()
    df_dims = pd.DataFrame()
    print("No facts found")


Processing Filings (2014-2025):   0%|          | 0/565 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
  -> No XBRL contexts found in this file. Attempting to download missing XML instance...
    -> Downloading missing XBRL from SEC: FilingSummary.xml
  -> Successfully downloaded and loaded TRMB_0000864749-18-000010_trmb201710k.xml!

Parsing iXBRL/XBRL document: TRMB_0000864749-17-000007_trmb201610k.htm...
  -> No XBRL contexts found in this file. Attempting to download missing XML instance...
    -> Downloading missing XBRL from SEC: FilingSummary.xml
  -> Successfully downloaded and loaded TRMB_0000864749-17-000007_trmb201610k.xml!

Parsing iXBRL/XBRL document: TRMB_0000864749-16-000092_trmb201510k.htm...
  -> No XBRL contexts found in this file. Attempting to download missing XML instance...
    -> Downloading missing XBRL from SEC: FilingSummary.xml
  -> Successfully downloaded and loaded TRMB_0000864749-16-000092_trmb201510k.xml!

Parsing iXBRL/XBRL document: TRMB_0000864749-15-000002_trmb201410k.htm...
  -> No XBRL contexts found 

In [ ]:
# filter for rows with "Revenue" in Metric, as well as "China" in Geographic Region
df_facts_revenue = df_facts[df_facts['Metric'].str.contains('Revenue', case=False, na=False)]
subset_columns = ["Metric", "Geographic Region (Raw XBRL)", "Reporting Period", "Reported Value", "Unit"]
df_facts_revenue_unique = df_facts_revenue.drop_duplicates(subset=subset_columns)
df_china_revenue = df_facts_revenue_unique[
    df_facts_revenue_unique['Geographic Region (Raw XBRL)'].str.contains(r'China|:CN\b|\bCN\b', case=False, na=False)
]

,Ticker,Source File,Matched Dimension,Metric,Geographic Region (Raw XBRL),Reporting Period,Reported Value,Calculated Absolute Value,Unit,Year
0,ABT,ABT_0001628280-25-007110_abt-20241231.htm,srt:StatementGeographicalAxis,us-gaap:RevenueFromContractWithCustomerExcludi...,us-gaap:NonUsMember,2024-01-01 to 2024-12-31,3858,3.858000e+09,usd,2024
1,ABT,ABT_0001628280-25-007110_abt-20241231.htm,srt:StatementGeographicalAxis,us-gaap:RevenueFromContractWithCustomerExcludi...,us-gaap:NonUsMember,2023-01-01 to 2023-12-31,3807,3.807000e+09,usd,2023
2,ABT,ABT_0001628280-25-007110_abt-20241231.htm,srt:StatementGeographicalAxis,us-gaap:RevenueFromContractWithCustomerExcludi...,us-gaap:NonUsMember,2022-01-01 to 2022-12-31,3766,3.766000e+09,usd,2022
3,ABT,ABT_0001628280-25-007110_abt-20241231.htm,srt:StatementGeographicalAxis,us-gaap:RevenueFromContractWithCustomerExcludi...,us-gaap:NonUsMember,2024-01-01 to 2024-12-31,1336,1.336000e+09,usd,2024
4,ABT,ABT_0001628280-25-007110_abt-20241231.htm,srt:StatementGeographicalAxis,us-gaap:RevenueFromContractWithCustomerExcludi...,us-gaap:NonUsMember,2023-01-01 to 2023-12-31,1259,1.259000e+09,usd,2023
5,ABT,ABT_0001628280-25-007110_abt-20241231.htm,srt:StatementGeographicalAxis,us-gaap:RevenueFromContractWithCustomerExcludi...,us-gaap:NonUsMember,2022-01-01 to 2022-12-31,1146,1.146000e+09,usd,2022
6,ABT,ABT_0001628280-25-007110_abt-20241231.htm,srt:StatementGeographicalAxis,us-gaap:RevenueFromContractWithCustomerExcludi...,us-gaap:NonUsMember,2024-01-01 to 2024-12-31,5194,5.194000e+09,usd,2024
7,ABT,ABT_0001628280-25-007110_abt-20241231.htm,srt:StatementGeographicalAxis,us-gaap:RevenueFromContractWithCustomerExcludi...,us-gaap:NonUsMember,2023-01-01 to 2023-12-31,5066,5.066000e+09,usd,2023
8,ABT,ABT_0001628280-25-007110_abt-20241231.htm,srt:StatementGeographicalAxis,us-gaap:RevenueFromContractWithCustomerExcludi...,us-gaap:NonUsMember,2022-01-01 to 2022-12-31,4912,4.912000e+09,usd,2022
9,ABT,ABT_0001628280-25-007110_abt-20241231.htm,srt:StatementGeographicalAxis,us-gaap:RevenueFromContractWithCustomerExcludi...,country:US,2024-01-01 to 2024-12-31,2208,2.208000e+09,usd,2024



Total rows found containing 'Revenue': 2189
